# RecRec: Recursive Refinement for Sequential Recommendation
PyTorch Lightning Implementation matching the paper specifications and architecture.

### Key Architectural Concepts:
- **Semantic Item Representation**: Pretrained SBERT sentence embeddings ($\mathbf{e} \in \mathbb{R}^{384}$) derived from textual item metadata.
- **Input Encoding Layer**: Masked mean aggregation of interaction histories $\mathbf{x} = \frac{1}{|S|} \sum_{i \in S} \mathbf{e}_i$, initializing user preference $\mathbf{y}_0 = \mathbf{x}$ and recursive latent state $\mathbf{z}_0 = \mathbf{0}$.
- **Shared Recursive Core**: Nonlinear transformation $f_\phi$ shared across both inner recursive loops and outer refinement steps.
- **Evidence-Anchored Correction**: Gated residual updating $\mathbf{z}_t = (1 - \mathbf{g}_t) \odot \mathbf{z}_t^{(n)} + \mathbf{g}_t \odot \mathbf{x}$ to prevent semantic drift.
- **Multi-Step Deep Supervision**: Averaged cross-entropy loss over all $T$ refinement trajectories $\mathcal{L} = \frac{1}{T} \sum_{t=1}^T \mathcal{L}_{\text{CE}}(\mathbf{y}_t, i^*)$.
- **Exponential Moving Average (EMA)**: Weight decay $\beta = 0.999$ during training for stable inference.

In [ ]:
!uv pip install pytorch_lightning sentence-transformers -q

In [ ]:
from collections import defaultdict
from collections.abc import Sequence
from dataclasses import dataclass
import math
from pathlib import Path
import pickle
import random

import numpy as np
import pytorch_lightning as pl
from pytorch_lightning.callbacks import Callback
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

## 1. Reproducibility & Seeds
Set deterministic random seeds across Python, NumPy, and PyTorch for reproducible runs.

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


SEED = 42
set_seed(SEED)

## 2. Configuration (`RecRecConfig`)
Hyperparameter configuration matching the paper settings:
- $d = 384$ (SBERT embedding dimension)
- Max sequence length = 50
- Outer refinement steps $T = 7$
- Inner recursive steps $n = 3$
- Core MLP depth = 5
- Candidate set size = 100 (1 target + 99 sampled negatives)
- Optimization: Adam ($\text{lr} = 1\times 10^{-3}$, batch size = 512, 50 epochs, EMA decay = 0.999)

In [ ]:
@dataclass
class RecRecConfig:
    embedding_dim: int = 384
    max_history_length: int = 50
    outer_steps: int = 7
    inner_steps: int = 3
    core_depth: int = 5
    preference_scale: float = 1.0  # L in Eq. (3)
    temperature: float = 1.0  # tau in Eq. (4)
    candidate_size: int = 100
    learning_rate: float = 1e-3
    batch_size: int = 512
    max_epochs: int = 50
    ema_decay: float = 0.999
    freeze_item_embeddings: bool = False
    num_workers: int = 3
    exclude_history_items_from_negatives: bool = True


CONFIG = RecRecConfig()

## 3. Data Loading & Index Validation
Loads sequential user-item interactions and verifies contiguous $0$-based item index alignment with the embedding table.

In [ ]:
def load_user_sequences(interaction_path: str | Path) -> dict[int, list[int]]:
    sequences: dict[int, list[int]] = defaultdict(list)

    with open(interaction_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            user_id, item_id = map(int, parts[:2])
            sequences[user_id].append(item_id)

    sequences = {uid: seq for uid, seq in sequences.items() if len(seq) >= 2}
    if not sequences:
        raise ValueError("No valid user sequences found.")
    return sequences


def validate_item_indexing(user_sequences: dict[int, list[int]], num_items: int) -> None:
    ids = [item_id for seq in user_sequences.values() for item_id in seq]
    if not ids:
        raise ValueError("No item IDs found.")
    if min(ids) < 0:
        raise ValueError(f"Found negative item ID: {min(ids)}")
    if max(ids) >= num_items:
        raise ValueError(
            f"Maximum item ID {max(ids)} exceeds embedding table size {num_items}."
        )

## 4. Semantic Embedding Extraction (SBERT)
Extracts L2-normalized 384-dimensional sentence embeddings from item metadata titles using `all-MiniLM-L6-v2`.

In [ ]:
def extract_sbert_item_embeddings(
    interaction_path: str | Path,
    metadata_path: str | Path,
    output_path: str | Path,
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    batch_size: int = 512,
    device: str | None = None,
) -> torch.Tensor:
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    user_sequences = load_user_sequences(interaction_path)
    item_ids = sorted({i for seq in user_sequences.values() for i in seq})

    expected_ids = list(range(len(item_ids)))
    if item_ids != expected_ids:
        raise ValueError("Item IDs must be contiguous 0..N-1 before SBERT extraction.")

    with open(metadata_path, "rb") as f:
        metadata = pickle.load(f)

    id_to_title = (
        metadata["title"] if isinstance(metadata, dict) and "title" in metadata else metadata
    )

    texts = []
    for item_id in item_ids:
        text = id_to_title.get(item_id, "unknown item")
        if text is None or not str(text).strip():
            text = "unknown item"
        texts.append(str(text))

    sbert = SentenceTransformer(model_name).to(device)

    chunks = []
    for start in tqdm(range(0, len(texts), batch_size), desc="SBERT"):
        batch_texts = texts[start:start + batch_size]
        chunks.append(
            sbert.encode(
                batch_texts,
                convert_to_tensor=True,
                device=device,
                normalize_embeddings=True,
            )
        )

    item_embeddings = torch.cat(chunks, dim=0).to(device)

    if item_embeddings.shape != (len(item_ids), 384):
        raise RuntimeError(
            f"Expected ({len(item_ids)}, 384), got {tuple(item_embeddings.shape)}"
        )

    if not torch.isfinite(item_embeddings).all():
        raise RuntimeError("SBERT embedding matrix contains non-finite values.")

    torch.save(item_embeddings, output_path)
    return item_embeddings

## 5. Leave-One-Out Task Construction
Constructs causal training sequences and strictly leave-one-out validation targets:
- **Validation**: Predict the final interaction $i_{|S|}$ given history $i_1, \dots, i_{|S|-1}$.
- **Training**: Causal prefixes $(i_1, \dots, i_k) \to i_{k+1}$ for $1 \le k < |S|-1$.

In [ ]:
def make_train_val_pairs(
    user_sequences: dict[int, list[int]],
) -> tuple[list[tuple[list[int], int]], list[tuple[list[int], int]]]:
    train_pairs = []
    val_pairs = []

    for seq in user_sequences.values():
        if len(seq) < 3:
            continue

        # Leave-one-out validation target
        val_pairs.append((seq[:-1], seq[-1]))

        # Causal prefixes for training
        for i in range(1, len(seq) - 1):
            train_pairs.append((seq[:i], seq[i]))

    return train_pairs, val_pairs

## 6. Candidate Negative Sampling & Data Pipeline
Samples 99 negative items uniformly without replacement (excluding history and target) and collates padded interaction batches.

In [ ]:
def sample_candidate_set(
    target_item: int,
    history: Sequence[int],
    num_items: int,
    candidate_size: int,
    exclude_history: bool = True,
) -> tuple[list[int], int]:
    forbidden = {target_item}
    if exclude_history:
        forbidden.update(history)

    available = [i for i in range(num_items) if i not in forbidden]
    if len(available) < candidate_size - 1:
        raise ValueError("Not enough negatives to construct candidate set.")

    negatives = random.sample(available, candidate_size - 1)
    candidates = negatives + [target_item]
    random.shuffle(candidates)
    return candidates, candidates.index(target_item)


class RecRecDataset(Dataset):
    def __init__(self, pairs: Sequence[tuple[Sequence[int], int]]):
        self.pairs = [(list(h), int(t)) for h, t in pairs]

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> tuple[list[int], int]:
        return self.pairs[index]


class RecRecCollator:
    def __init__(self, config: RecRecConfig, num_items: int):
        self.max_history_length = config.max_history_length
        self.candidate_size = config.candidate_size
        self.num_items = num_items
        self.exclude_history = config.exclude_history_items_from_negatives

    def __call__(
        self, batch: list[tuple[list[int], int]]
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        batch_size = len(batch)

        history_ids = torch.zeros(
            batch_size, self.max_history_length, dtype=torch.long
        )
        history_mask = torch.zeros(
            batch_size, self.max_history_length, dtype=torch.float32
        )
        candidate_ids = torch.zeros(
            batch_size, self.candidate_size, dtype=torch.long
        )
        target_index = torch.zeros(batch_size, dtype=torch.long)

        for row, (history, target) in enumerate(batch):
            history = history[-self.max_history_length:]
            length = len(history)

            history_ids[row, -length:] = torch.tensor(
                history, dtype=torch.long
            )
            history_mask[row, -length:] = 1.0

            candidates, target_position = sample_candidate_set(
                target_item=target,
                history=history,
                num_items=self.num_items,
                candidate_size=self.candidate_size,
                exclude_history=self.exclude_history,
            )

            candidate_ids[row] = torch.tensor(
                candidates, dtype=torch.long
            )
            target_index[row] = target_position

        return history_ids, history_mask, candidate_ids, target_index


class RecRecDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_pairs: Sequence[tuple[Sequence[int], int]],
        val_pairs: Sequence[tuple[Sequence[int], int]],
        num_items: int,
        config: RecRecConfig,
    ):
        super().__init__()
        self.train_pairs = list(train_pairs)
        self.val_pairs = list(val_pairs)
        self.num_items = num_items
        self.config = config
        self.collator = RecRecCollator(config, num_items)

    def setup(self, stage: str | None = None) -> None:
        self.train_dataset = RecRecDataset(self.train_pairs)
        self.val_dataset = RecRecDataset(self.val_pairs)

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            self.train_dataset,
            batch_size=self.config.batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
            pin_memory=torch.cuda.is_available(),
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            self.val_dataset,
            batch_size=self.config.batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
            pin_memory=torch.cuda.is_available(),
        )

## 7. Input Encoding Layer
Aggregates past item embeddings via masked mean:
$$\mathbf{x} = \frac{\sum_{i \in S} \mathbf{e}_i}{|S|}, \quad \mathbf{y}_0 = \mathbf{x}, \quad \mathbf{z}_0 = \mathbf{0}$$

In [ ]:
class InputEncoding(nn.Module):
    def forward(
        self,
        item_weight: torch.Tensor,
        history_item_ids: torch.Tensor,
        history_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        history_embeddings = F.embedding(history_item_ids, item_weight)
        mask = history_mask.to(history_embeddings.dtype).unsqueeze(-1)

        denominator = mask.sum(dim=1).clamp_min(1e-12)
        x = (history_embeddings * mask).sum(dim=1) / denominator

        y0 = x
        z0 = torch.zeros_like(x)
        return x, y0, z0

## 8. Shared Recursive Core ($f_\phi$)
Nonlinear multilayer perceptron with LayerNorm and ReLU activations:
- Input: Concatenation $[\mathbf{x} \,\|\, \mathbf{y} \,\|\, \mathbf{z}] \in \mathbb{R}^{3d}$
- Hidden/Output: Dimension $d = 384$ with depth $D = 5$.

In [ ]:
class CoreRecursionMLP(nn.Module):
    def __init__(self, embedding_dim: int, depth: int):
        super().__init__()
        if depth < 1:
            raise ValueError(f"depth must be >= 1, got {depth}")

        layers = []
        for layer_idx in range(depth):
            in_dim = 3 * embedding_dim if layer_idx == 0 else embedding_dim
            layers.append(nn.Linear(in_dim, embedding_dim))
            layers.append(nn.LayerNorm(embedding_dim))
            if layer_idx < depth - 1:
                layers.append(nn.ReLU())

        self.network = nn.Sequential(*layers)

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return self.network(state)

## 9. Recursive Preference Refinement
Implements the core recurrent reasoning equations across $T$ refinement steps:
- **Inner Recursion** (Eq. 1):
  $$\mathbf{z}_t^{(j)} = f_\phi([\mathbf{x} \,\|\, \mathbf{y}_t \,\|\, \mathbf{z}_t^{(j-1)}]), \quad j=1,\dots,n$$
- **Evidence-Anchored Correction** (Eq. 2):
  $$\mathbf{g}_t = \sigma(\mathbf{W}_t [\mathbf{x} \,\|\, \mathbf{y}_t]), \quad \mathbf{z}_t = (1 - \mathbf{g}_t) \odot \mathbf{z}_t^{(n)} + \mathbf{g}_t \odot \mathbf{x}$$
- **Preference Update** (Eq. 3):
  $$\mathbf{y}_{t+1} = \mathbf{y}_t + L \cdot \tanh(f_\phi([\mathbf{x} \,\|\, \mathbf{y}_t \,\|\, \mathbf{z}_t]))$$

In [ ]:
class RecursivePreferenceRefinement(nn.Module):
    def __init__(self, config: RecRecConfig):
        super().__init__()
        d = config.embedding_dim
        self.outer_steps = config.outer_steps
        self.inner_steps = config.inner_steps
        self.preference_scale = config.preference_scale

        self.f_phi = CoreRecursionMLP(embedding_dim=d, depth=config.core_depth)
        self.correction_gates = nn.ModuleList(
            [nn.Linear(2 * d, d) for _ in range(config.outer_steps)]
        )

    def forward(
        self,
        x: torch.Tensor,
        y0: torch.Tensor,
        z0: torch.Tensor,
    ) -> list[torch.Tensor]:
        y = y0
        z = z0
        y_states = []

        for t in range(self.outer_steps):
            z_inner = z

            # Eq. (1): Inner recursive reasoning
            for _ in range(self.inner_steps):
                z_inner = self.f_phi(torch.cat([x, y, z_inner], dim=-1))

            # Eq. (2): Gated evidence correction
            g = torch.sigmoid(self.correction_gates[t](torch.cat([x, y], dim=-1)))
            z = (1.0 - g) * z_inner + g * x

            # Eq. (3): Preference update
            delta = torch.tanh(self.f_phi(torch.cat([x, y, z], dim=-1)))
            y = y + self.preference_scale * delta
            y_states.append(y)

        return y_states

## 10. Candidate Scoring Layer
Computes scaled dot-product logits for candidate items at each refinement step $t$:
$$s_t(j) = \frac{\mathbf{y}_{t+1}^\top \mathbf{e}_j}{\tau}$$

In [ ]:
class CandidateScoring(nn.Module):
    def __init__(self, temperature: float):
        super().__init__()
        if temperature <= 0:
            raise ValueError(f"temperature must be positive, got {temperature}")
        self.temperature = temperature

    def forward(
        self,
        item_weight: torch.Tensor,
        y_states: list[torch.Tensor],
        candidate_ids: torch.Tensor,
    ) -> list[torch.Tensor]:
        candidate_embeddings = F.embedding(candidate_ids, item_weight)
        logits = []

        for y in y_states:
            scores = torch.einsum("bd,bnd->bn", y, candidate_embeddings)
            logits.append(scores / self.temperature)

        return logits

## 11. Complete RecRec Architecture
Encapsulates embedding lookup, input encoding, recursive refinement, and candidate scoring into a unified `nn.Module`.

In [ ]:
class RecRec(nn.Module):
    def __init__(self, pretrained_sbert_embeddings: torch.Tensor, config: RecRecConfig):
        super().__init__()

        if pretrained_sbert_embeddings.ndim != 2:
            raise ValueError(
                f"SBERT embeddings must be 2D [num_items, dim], got shape {pretrained_sbert_embeddings.shape}"
            )

        if pretrained_sbert_embeddings.size(1) != config.embedding_dim:
            raise ValueError(
                f"Expected embedding dim {config.embedding_dim}, got {pretrained_sbert_embeddings.size(1)}"
            )

        self.item_embeddings = nn.Embedding.from_pretrained(
            pretrained_sbert_embeddings.float(),
            freeze=config.freeze_item_embeddings,
        )

        self.input_encoding = InputEncoding()
        self.preference_refinement = RecursivePreferenceRefinement(config)
        self.candidate_scoring = CandidateScoring(config.temperature)

    @property
    def item_weight(self) -> torch.Tensor:
        return self.item_embeddings.weight

    def forward(
        self,
        history_ids: torch.Tensor,
        history_mask: torch.Tensor,
        candidate_ids: torch.Tensor | None = None,
    ) -> list[torch.Tensor] | torch.Tensor:
        x, y0, z0 = self.input_encoding(self.item_weight, history_ids, history_mask)
        y_states = self.preference_refinement(x, y0, z0)

        if candidate_ids is None:
            return y_states[-1]

        return self.candidate_scoring(self.item_weight, y_states, candidate_ids)

## 12. Deep Supervision Loss
Computes the averaged multi-step cross-entropy loss over all $T$ outer refinement iterations:
$$\mathcal{L}_{\text{total}} = \frac{1}{T} \sum_{t=1}^T \mathcal{L}_{\text{CE}}(\mathbf{s}_t, i^*)$$

In [ ]:
def deep_supervision_loss(
    logits_per_step: list[torch.Tensor], target_index: torch.Tensor
) -> torch.Tensor:
    return torch.stack(
        [F.cross_entropy(logits, target_index) for logits in logits_per_step]
    ).mean()

## 13. Exponential Moving Average (EMA) Callback
Maintains exponential moving averages of model parameters during training ($\beta = 0.999$) and swaps weights for validation and testing.

In [ ]:
class EMACallback(Callback):
    def __init__(self, decay: float = 0.999):
        super().__init__()
        self.decay = decay
        self.shadow: dict[str, torch.Tensor] = {}
        self.backup: dict[str, torch.Tensor] = {}

    def trainable_parameters(self, module: torch.nn.Module):
        return ((name, p) for name, p in module.named_parameters() if p.requires_grad)

    def initialize(self, module: torch.nn.Module) -> None:
        if not self.shadow:
            self.shadow = {
                name: p.detach().clone()
                for name, p in self.trainable_parameters(module)
            }

    @torch.no_grad()
    def on_train_start(self, trainer, pl_module) -> None:
        self.initialize(pl_module)

    @torch.no_grad()
    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx) -> None:
        self.initialize(pl_module)

        for name, p in self.trainable_parameters(pl_module):
            if name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(p.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def apply_shadow(self, pl_module: torch.nn.Module) -> None:
        if not self.shadow:
            return

        self.backup = {}
        for name, p in self.trainable_parameters(pl_module):
            if name in self.shadow:
                self.backup[name] = p.detach().clone()
                p.copy_(self.shadow[name])

    @torch.no_grad()
    def restore(self, pl_module: torch.nn.Module) -> None:
        if not self.backup:
            return

        for name, p in self.trainable_parameters(pl_module):
            if name in self.backup:
                p.copy_(self.backup[name])
        self.backup.clear()

    def on_validation_epoch_start(self, trainer, pl_module) -> None:
        self.apply_shadow(pl_module)

    def on_validation_epoch_end(self, trainer, pl_module) -> None:
        self.restore(pl_module)

    def on_test_epoch_start(self, trainer, pl_module) -> None:
        self.apply_shadow(pl_module)

    def on_test_epoch_end(self, trainer, pl_module) -> None:
        self.restore(pl_module)

## 14. Ranking Evaluation Metrics
Computes standard Top-$K$ ranking metrics:
- **Hit Ratio ($HR@K$)**: $\mathbb{I}(\text{rank} \le K)$
- **NDCG@$K$**: $\frac{\mathbb{I}(\text{rank} \le K)}{\log_2(\text{rank} + 1)}$
- **Precision@$K$**: $\frac{\mathbb{I}(\text{rank} \le K)}{K}$

In [ ]:
def compute_ranking_metrics(ranks: torch.Tensor) -> dict[str, float]:
    ranks = ranks.float()
    metrics = {}

    for k in (1, 5, 10):
        hit = ranks <= k
        metrics[f"HR@{k}"] = hit.float().mean().item()
        metrics[f"NDCG@{k}"] = torch.where(
            hit, 1.0 / torch.log2(ranks + 1.0), torch.zeros_like(ranks)
        ).mean().item()
        metrics[f"Prec@{k}"] = (hit.float() / float(k)).mean().item()

    return metrics

## 15. PyTorch Lightning Module
Encapsulates training step optimization under deep supervision and full leave-one-out ranking evaluation at validation time.

In [ ]:
class RecRecLightning(pl.LightningModule):
    def __init__(self, pretrained_sbert_embeddings: torch.Tensor, config: RecRecConfig):
        super().__init__()
        self.save_hyperparameters(ignore=["pretrained_sbert_embeddings"])
        self.config = config
        self.model = RecRec(pretrained_sbert_embeddings, config)
        self.validation_ranks: list[torch.Tensor] = []

    def forward(
        self,
        history_ids: torch.Tensor,
        history_mask: torch.Tensor,
        candidate_ids: torch.Tensor | None = None,
    ) -> list[torch.Tensor] | torch.Tensor:
        return self.model(history_ids, history_mask, candidate_ids)

    def training_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> torch.Tensor:
        history_ids, history_mask, candidate_ids, target_index = batch
        logits = self(history_ids, history_mask, candidate_ids)
        loss = deep_supervision_loss(logits, target_index)

        self.log(
            "train_loss",
            loss,
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            batch_size=history_ids.size(0),
        )
        return loss

    def on_validation_epoch_start(self) -> None:
        self.validation_ranks = []

    @torch.no_grad()
    def validation_step(
        self,
        batch: tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
        batch_idx: int,
    ) -> None:
        history_ids, history_mask, candidate_ids, target_index = batch
        logits = self(history_ids, history_mask, candidate_ids)

        final_logits = logits[-1]
        ranking = torch.argsort(final_logits, dim=1, descending=True)
        ranks = (ranking == target_index.unsqueeze(1)).nonzero(as_tuple=True)[1] + 1
        self.validation_ranks.append(ranks.to(self.device))

    def on_validation_epoch_end(self) -> None:
        if not self.validation_ranks:
            return

        ranks = torch.cat(self.validation_ranks)
        metrics = compute_ranking_metrics(ranks)
        self.validation_ranks.clear()

        for name, value in metrics.items():
            self.log(
                f"val_{name.lower().replace('@', '')}",
                value,
                prog_bar=(name == "NDCG@10"),
            )

    def configure_optimizers(self) -> torch.optim.Optimizer:
        return torch.optim.Adam(self.parameters(), lr=self.config.learning_rate)

## 16. Paper-Compliance Validation
Verifies that all model parameters, shared core instances, and gate structures strictly adhere to the specification.

In [ ]:
def validate_paper_compliance(model: RecRecLightning, config: RecRecConfig) -> None:
    assert config.embedding_dim == 384
    assert config.max_history_length == 50
    assert config.outer_steps == 7
    assert config.inner_steps == 3
    assert config.candidate_size == 100
    assert config.learning_rate == 1e-3
    assert config.batch_size == 512
    assert config.max_epochs == 50
    assert config.ema_decay == 0.999
    assert config.freeze_item_embeddings is False

    item_table = model.model.item_embeddings
    assert item_table.weight.requires_grad is True

    refinement = model.model.preference_refinement
    assert len(refinement.correction_gates) == config.outer_steps
    assert isinstance(refinement.f_phi, CoreRecursionMLP)

## 17. Pre-Training Numerical Diagnostics
Executes a forward pass on a single batch to verify loss scales against theoretical uniform random baselines ($\ln(100) \approx 4.605$).

In [ ]:
@torch.no_grad()
def inspect_one_batch(model: RecRecLightning, loader: DataLoader) -> None:
    device = next(model.parameters()).device

    batch = next(iter(loader))
    batch = [x.to(device) for x in batch]
    history_ids, history_mask, candidate_ids, target_index = batch

    logits_per_step = model(history_ids, history_mask, candidate_ids)
    losses = [F.cross_entropy(logits, target_index).item() for logits in logits_per_step]

    final_logits = logits_per_step[-1]
    ranks = (
        (torch.argsort(final_logits, dim=1, descending=True) == target_index.unsqueeze(1))
        .nonzero(as_tuple=True)[1] + 1
    )

    print("One-batch diagnostic:")
    print(f"Step losses: {[round(x, 4) for x in losses]}")
    print(f"Mean loss: {sum(losses) / len(losses):.4f}")
    print(f"Mean target rank: {ranks.float().mean().item():.2f}")
    print(f"HR@1: {(ranks <= 1).float().mean().item():.4f}")
    print(f"HR@10: {(ranks <= 10).float().mean().item():.4f}")
    print(f"Uniform 100-way CE: {math.log(100):.4f}")

## 18. Dataset Paths
Configured for Kaggle / Local environment execution.

In [ ]:
INTERACTION_PATH = "/kaggle/input/datasets/chrisolande2/recsys/data/Luxury_Beauty_5.txt"
SBERT_EMBEDDING_PATH = "/kaggle/input/datasets/chrisolande2/recsys/data/sbert_item_embeddings.pt"

# Fallback to local data paths if running outside Kaggle
if not Path(INTERACTION_PATH).exists():
    INTERACTION_PATH = "data/Luxury_Beauty_5.txt"
    SBERT_EMBEDDING_PATH = "data/sbert_item_embeddings.pt" 

## 19. Load Sequences & Item Embeddings

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
user_sequences = load_user_sequences(INTERACTION_PATH)
item_embeddings = torch.load(SBERT_EMBEDDING_PATH, map_location=device)

num_items = item_embeddings.size(0)
validate_item_indexing(user_sequences, num_items)

print(f"Users: {len(user_sequences)}")
print(f"Items: {num_items}")
print(f"Interactions: {sum(len(s) for s in user_sequences.values())}")
print(f"SBERT shape: {tuple(item_embeddings.shape)}")

## 20. Construct Train / Validation Sequences

In [ ]:
train_pairs, val_pairs = make_train_val_pairs(user_sequences)

print(f"Train instances: {len(train_pairs)}")
print(f"Val instances: {len(val_pairs)}")

## 21. Initialize Lightning DataModule

In [ ]:
datamodule = RecRecDataModule(
    train_pairs=train_pairs,
    val_pairs=val_pairs,
    num_items=num_items,
    config=CONFIG,
)

## 22. Initialize Model & Validate Architecture

In [ ]:
model = RecRecLightning(
    pretrained_sbert_embeddings=item_embeddings,
    config=CONFIG,
).to(device)

validate_paper_compliance(model, CONFIG)
print("Paper-compliance checks passed successfully.")

## 23. Run Pre-Training Numerical Diagnostic

In [ ]:
datamodule.setup("fit")
inspect_one_batch(
    model,
    datamodule.train_dataloader(),
)

## 24. Initialize EMA Callback & Trainer

In [ ]:
ema_callback = EMACallback(decay=CONFIG.ema_decay)

trainer = pl.Trainer(
    max_epochs=CONFIG.max_epochs,
    accelerator="auto",
    devices="auto",
    callbacks=[ema_callback],
    enable_progress_bar=True,
)

## 25. Train RecRec Model

In [ ]:
trainer.fit(
    model,
    datamodule=datamodule,
)

## 26. Evaluate Ranking Performance on Validation Split

In [ ]:
trainer.validate(
    model,
    datamodule=datamodule,
)